# AnDet Pro on Google Colab

This notebook runs the AnDet Pro experiment on Google Colab and saves the results in Google Drive.
Run the cells from top to bottom. To run a cell, click it and press Shift + Enter.

**Before you start**
1. In Google Drive, create a folder named `AnDet`.
2. Put these two files in that folder: `AnDet_Pro_v3_rebuilt.zip` and `NF-CSE-CIC-IDS2018.parquet`.
3. In the Colab menu go to Runtime, then Change runtime type, choose T4 GPU and click Save. The GPU makes training faster.

## Step 1: Connect Google Drive and unpack the project

This cell connects Google Drive, checks that the two files are in the AnDet folder, unzips the project, and copies the data file to Colab's own disk because it reads faster there. It ends by printing the project folder, `/content/AnDet_Pro`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, zipfile
SRC = '/content/drive/MyDrive/AnDet'
DATA = f'{SRC}/NF-CSE-CIC-IDS2018.parquet'
RESULTS = f'{SRC}/results'
assert os.path.exists(f'{SRC}/AnDet_Pro_v3_rebuilt.zip'), 'Put AnDet_Pro_v3_rebuilt.zip in the Drive folder AnDet'
assert os.path.exists(DATA), 'Put NF-CSE-CIC-IDS2018.parquet in the Drive folder AnDet'
if not os.path.exists('/content/AnDet_Pro'):
    zipfile.ZipFile(f'{SRC}/AnDet_Pro_v3_rebuilt.zip').extractall('/content')
shutil.copy(DATA, '/content/NF-CSE-CIC-IDS2018.parquet')   # a local copy reads faster than Drive
DATA_LOCAL = '/content/NF-CSE-CIC-IDS2018.parquet'
os.makedirs(RESULTS, exist_ok=True)
%cd /content/AnDet_Pro

Mounted at /content/drive
/content/AnDet_Pro


## Step 2: Install pytest and check the set-up

Colab already has TensorFlow, NumPy, pandas, SciPy, PyArrow and Matplotlib, so only pytest needs to be installed.
Do not run `pip install -r requirements.txt` on Colab, because it can replace the TensorFlow that Colab already has.

The last line of the output must say **READY**. A note about the GPU is fine.

In [2]:
!pip -q install pytest
!python check_setup.py --dataset {DATA_LOCAL}

AnDet Pro - set-up check
------------------------------------------------------------
[NOTE] Python 3.13.16 on Linux x86_64
       -> TensorFlow may not have a build for this Python yet. Python 3.11 is the safe choice.
[PASS] numpy 2.1.3
[PASS] pandas 2.2.3
[PASS] scipy 1.16.3
[PASS] pyarrow 23.0.1
[PASS] matplotlib 3.10.0
[PASS] pytest 8.4.2
[PASS] TensorFlow 2.21.0 (loaded in 3.9 s)
[PASS] GPU found: /physical_device:GPU:0
[PASS] Data file NF-CSE-CIC-IDS2018.parquet: 62 MB, 5,918,803 rows
[PASS] Label column: Label; attack-type column: not found; 10 features
       features: L4_SRC_PORT, L4_DST_PORT, PROTOCOL, L7_PROTO, IN_BYTES, OUT_BYTES, IN_PKTS, OUT_PKTS, TCP_FLAGS, FLOW_DURATION_MILLISECONDS
[PASS] config/config.ini is valid
       what each mode runs:
       mode       every n-th flow  max epochs  trials per condition  models trained
       fast                   590           4                    30               1
       standard                59         100                 

## Step 3: Quick test run

This is a small run that takes a few minutes. It only checks that the whole program works.
Do not use its numbers in the thesis or the slides.

When it finishes, the output shows `THIS WAS A FAST CHECK`. If you see `SOMETHING WENT WRONG`, stop and fix that first.

In [3]:
!PYTHONUNBUFFERED=1 python run.py --mode fast --dataset {DATA_LOCAL} --results-dir {RESULTS}

AnDet Pro   mode: FAST   started 2026-10-29 02:15:10
FAST CHECK: small sample, few epochs. It tests that the program works.
Do NOT put these numbers in the thesis or the slides. Use  --mode standard  for that.
  data file         : /content/NF-CSE-CIC-IDS2018.parquet
  keep every        : 590-th flow of the file
  models to train   : 1  (at most 4 epochs each, stops early if the validation loss stops improving)
  trials            : up to 10 attack scenarios x 3 repetitions = up to 30 trials per condition
  random seed       : 42
  extra checks      : VMD off=no, block VMD=no, window sweep='', extra seeds=0, density sweep=''
  results folder    : /content/drive/MyDrive/AnDet/results/20261029_021510_fast


=== 1. Read and split the data ===
  file: NF-CSE-CIC-IDS2018.parquet  rows in file 5,918,803  after sampling 10,032 (every 590-th)
  features (10): L4_SRC_PORT, L4_DST_PORT, PROTOCOL, L7_PROTO, IN_BYTES, OUT_BYTES, IN_PKTS, OUT_PKTS, TCP_FLAGS, FLOW_DURATION_MILLISECONDS
  normal tra

## Step 4: The real run

This run gives the numbers for the thesis and the slides.

Training is limited to about 60 minutes (`model.max_minutes=60`). When the limit is reached the best epoch so far is kept, and the results file says so. If this happens, mention it in the thesis.
After the first epoch the output shows how many seconds one epoch takes, so you can see roughly how long the run will last.

Keep this browser tab open until the run ends. The run is finished when the output shows `Total time` and `WHAT TO DO NEXT`, and the summary box says `mode: standard`.

In [4]:
!PYTHONUNBUFFERED=1 python run.py --mode standard --dataset {DATA_LOCAL} --results-dir {RESULTS} --set model.max_minutes=60

AnDet Pro   mode: STANDARD   started 2026-10-29 02:16:41
  data file         : /content/NF-CSE-CIC-IDS2018.parquet
  keep every        : 59-th flow of the file
  models to train   : 2  (at most 100 epochs each, stops early if the validation loss stops improving)
  trials            : up to 10 attack scenarios x 30 repetitions = up to 300 trials per condition
  random seed       : 42
  extra checks      : VMD off=yes, block VMD=no, window sweep='', extra seeds=0, density sweep=''
  results folder    : /content/drive/MyDrive/AnDet/results/20261029_021641_standard


=== 1. Read and split the data ===
  file: NF-CSE-CIC-IDS2018.parquet  rows in file 5,918,803  after sampling 100,319 (every 59-th)
  features (10): L4_SRC_PORT, L4_DST_PORT, PROTOCOL, L7_PROTO, IN_BYTES, OUT_BYTES, IN_PKTS, OUT_PKTS, TCP_FLAGS, FLOW_DURATION_MILLISECONDS
  normal training flows 69,141  (fit 48,399 / validation 10,371 / calibration 10,371)
  test flows 30,096  (18.2% attack)
--- 1. Read and split the data: 1.2

## Step 5: If Colab disconnected during the real run

1. Run Step 1 and Step 2 again.
2. Open Google Drive, go to AnDet, then results, and copy the name of the newest folder (for example `20261029_021500_standard`).
3. Paste that name into the cell below and run it.

Models that already finished training are loaded, so they are not trained again.

In [7]:
RESUME = 'PUT_THE_FOLDER_NAME_HERE'   # for example 20261029_021500_standard
!PYTHONUNBUFFERED=1 python run.py --mode standard --dataset {DATA_LOCAL} --results-dir {RESULTS} --set model.max_minutes=60 --resume {RESULTS}/{RESUME}

Cannot resume: /content/drive/MyDrive/AnDet/results/PUT_THE_FOLDER_NAME_HERE does not exist.


## Step 6: Get the results

The results are already saved in Google Drive, in AnDet, then results, then the newest folder.
This cell also packs the newest run into one zip file and downloads it to your computer.

Open these two files first:
- `RESULTS_FOR_THESIS.md` has the tables and draft sentences for Chapter Four.
- `DOCUMENT_UPDATE_SHEET.md` lists every place in the thesis and the slides that must change.

Check that the first lines of `RESULTS_FOR_THESIS.md` say `Mode: standard`.

In [5]:
import glob
latest = open(f'{RESULTS}/LATEST.txt').read().strip()
print('Latest run:', latest)
shutil.make_archive('/content/andet_results', 'zip', latest)
from google.colab import files
files.download('/content/andet_results.zip')

Latest run: /content/drive/MyDrive/AnDet/results/20261029_021641_standard


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Optional: run the automated tests

This takes about two minutes. At the end it should say `158 passed`.

In [6]:
!python -m pytest tests -q

........................................................................ [ 45%]
........................................................................ [ 91%]
..............                                                           [100%]
=============================== warnings summary ===============================
tests/test_end_to_end.py::test_full_mode_runs_every_extra_check
  /content/AnDet_Pro/andet/analysis.py:448: RuntimeWarning: Mean of empty slice
    change_pct=_pct(np.nanmean(xa), np.nanmean(xb)))

tests/test_end_to_end.py::test_full_mode_runs_every_extra_check
  /content/AnDet_Pro/andet/analysis.py:461: RuntimeWarning: Mean of empty slice
    change_pct=_pct(np.nanmean(xa), np.nanmean(xb)))

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
158 passed, 2 warnings in 113.15s (0:01:53)
